<a href="https://colab.research.google.com/github/hardik-xi11/colab-notebooks-bckp/blob/main/pokemon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import json
from getpass import getpass

# 1. Prompt for credentials interactively
print("Enter your Kaggle credentials below:")
kaggle_username = input("Username: ")
kaggle_key = getpass("API Key (input will be hidden): ")

# 2. Create the credentials dictionary
token = {"username": kaggle_username, "key": kaggle_key}

# 3. Save it as kaggle.json in the correct folder
!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(token, file)

# 4. Set permissions (security requirement)
!chmod 600 ~/.kaggle/kaggle.json

# 5. Download the dataset
print("\n⬇️ Downloading dataset...")
!kaggle datasets download -d vishalsubbiah/pokemon-images-and-types --force
!unzip -q pokemon-images-and-types.zip -d dataset

print("✅ Success! Dataset downloaded and ready.")

Enter your Kaggle credentials below:
Username: xihardik05
API Key (input will be hidden): ··········

⬇️ Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/vishalsubbiah/pokemon-images-and-types
License(s): Attribution 4.0 International (CC BY 4.0)
  0% 0.00/3.68M [00:00<?, ?B/s]
100% 3.68M/3.68M [00:00<00:00, 736MB/s]
✅ Success! Dataset downloaded and ready.


In [3]:
df = pd.read_csv('dataset/pokemon.csv')

df['filename'] = df['Name'] + ".png"
df['path'] = "dataset/images/" + df['filename']

import os
df['exists'] = df['path'].apply(os.path.exists)
df_clean = df[df['exists']].copy()

print(f"Original entries: {len(df)}")
print(f"Valid images found: {len(df_clean)}")
display(df_clean.head(3))

Original entries: 809
Valid images found: 809


,Name,Type1,Type2,Evolution,filename,path,exists
0,bulbasaur,Grass,Poison,ivysaur,bulbasaur.png,dataset/images/bulbasaur.png,True
1,ivysaur,Grass,Poison,venusaur,ivysaur.png,dataset/images/ivysaur.png,True
2,venusaur,Grass,Poison,NaN,venusaur.png,dataset/images/venusaur.png,True


In [4]:
# Cell 3: Data Generators with Augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# We use heavy augmentation to teach the model to recognize the pokemon
# even if it's rotated, zoomed, or slightly shifted.
datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values
    rotation_range=20,        # Rotate images up to 20 degrees
    width_shift_range=0.1,    # Shift horizontally
    height_shift_range=0.1,   # Shift vertically
    shear_range=0.1,          # Slant the image
    zoom_range=0.1,           # Zoom in/out
    horizontal_flip=True,     # Flip left/right
    fill_mode='nearest'
)

# Create the generator
# "class_mode='categorical'" sets up the 809-way classification
train_generator = datagen.flow_from_dataframe(
    dataframe=df_clean,
    directory="dataset/images", # Folder containing images
    x_col="filename",
    y_col="Name",
    target_size=(224, 224),   # EfficientNet standard size
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

# Save the class labels (names) for later use
class_map = {v: k for k, v in train_generator.class_indices.items()}
num_classes = len(class_map)
print(f"Detected {num_classes} unique Pokemon classes.")

Found 809 validated image filenames belonging to 809 classes.
Detected 809 unique Pokemon classes.


In [5]:
# Cell 4: Build Professional Model Architecture
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# 1. Load the Base Model (Pre-trained on ImageNet)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the base model (optional, but good for small datasets)
base_model.trainable = False

# 3. Add Custom Layers
x = base_model.output
x = GlobalAveragePooling2D()(x)       # Condense features
x = Dropout(0.2)(x)                   # Prevent overfitting
predictions = Dense(num_classes, activation='softmax')(x) # Output layer (809 classes)

# 4. Compile
model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 5,085,900 (19.40 MB)

 Trainable params: 1,036,329 (3.95 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# Cell 5: Train
# We run for roughly 10-15 epochs to ensure it memorizes the forms
history = model.fit(
    train_generator,
    epochs=15,
    verbose=1
)

print("🎉 Training Complete!")

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.0000e+00 - loss: 7.1353
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 60s 2s/step - accuracy: 0.0000e+00 - loss: 6.9829
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 60s 2s/step - accuracy: 0.0000e+00 - loss: 6.8219
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 60s 2s/step - accuracy: 0.0000e+00 - loss: 6.8232
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step - accuracy: 0.0000e+00 - loss: 6.8152
Epoch 6/15
24/26 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.0030 - loss: 6.7880

In [ ]:
# Cell 6: Prediction System
from tensorflow.keras.preprocessing import image

def identify_pokemon(img_path):
    # 1. Load and Preprocess Image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normalize

    # 2. Make Prediction
    predictions = model.predict(img_array)

    # 3. Decode Results (Top 3)
    top_3_indices = np.argsort(predictions[0])[-3:][::-1]

    plt.imshow(img)
    plt.axis('off')
    plt.show()

    print("🔍 ANALYSIS RESULTS:")
    print("-" * 30)
    for i in top_3_indices:
        pokemon_name = class_map[i]
        confidence = predictions[0][i] * 100
        print(f"🔹 {pokemon_name.upper()}: {confidence:.2f}%")

# --- INTERACTIVE UPLOAD ---
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    identify_pokemon(filename)